# 07 — Training loop and PyTorch checks

Part of the micrograd repetition pack.


## Goal
Connect forward pass, loss, zeroing gradients, backward pass, and parameter updates into one training step.


In [ ]:
import math

_results = []

def check(name, condition, detail=""):
    ok = bool(condition)
    _results.append(ok)
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {name}" + (f" — {detail}" if detail else ""))

def close(a, b, tol=1e-6):
    return abs(a - b) <= tol

def summary():
    print(f"\nScore: {sum(_results)}/{len(_results)} tests passed")


In [ ]:
from math import exp, log, sin, cos

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1 / self.data) * out.grad
        out._backward = _backward
        return out

    def __pow__(self, power):
        assert isinstance(power, (int, float))
        out = Value(self.data ** power, (self,), f'**{power}')
        def _backward():
            self.grad += power * self.data ** (power - 1) * out.grad
        out._backward = _backward
        return out

    def sin(self):
        out = Value(sin(self.data), (self,), 'sin')
        def _backward():
            self.grad += cos(self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        e2x = (2 * self).exp()
        return (e2x - 1) / (e2x + 1)

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1
    def __rtruediv__(self, other): return Value(other) * self ** -1

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


In [ ]:
import random

class Module:
    def parameters(self): return []
    def zero_grad(self):
        for p in self.parameters(): p.grad = 0.0

class Neuron(Module):
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
    def __call__(self, x):
        if len(x) != len(self.w):
            raise ValueError(f"expected {len(self.w)} inputs, got {len(x)}")
        return (sum(wi * xi for wi, xi in zip(self.w, x)) + self.b).tanh()
    def parameters(self): return self.w + [self.b]

class Layer(Module):
    def __init__(self, nin, nout): self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]

class MLP(Module):
    def __init__(self, nin, nouts):
        sizes = [nin] + list(nouts)
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers: x = layer(x)
        return x
    def parameters(self): return [p for layer in self.layers for p in layer.parameters()]


In [ ]:
random.seed(1337)


In [ ]:
xs=[[2.0,3.0,-1.0],[3.0,-1.0,0.5],[0.5,1.0,1.0],[1.0,1.0,-1.0]]
ys=[1.0,-1.0,-1.0,1.0]

def squared_error(model, xs, ys):
    """TODO: return (loss, predictions)."""
    raise NotImplementedError

def train_step(model, xs, ys, learning_rate=0.01):
    """TODO: forward, zero gradients, backward, update; return loss value."""
    raise NotImplementedError


### Round A — One training step
Write the five phases in words before implementing them.


In [ ]:
_results.clear()
try:
    random.seed(1337); model=MLP(3,[4,4,1])
    before,_=squared_error(model,xs,ys)
    returned=train_step(model,xs,ys,0.01)
    after,_=squared_error(model,xs,ys)
    check("returns pre-update loss",close(returned,before.data))
    check("parameters received gradients",any(abs(p.grad)>0 for p in model.parameters()))
    check("one small step lowers loss",after.data<before.data)
except Exception as e: check("one training step",False,repr(e))
summary()


### Round B — Repeated optimization
Train for 50 steps and record the losses. Do not recreate the model inside the loop.


In [ ]:
def train(model, xs, ys, steps=50, learning_rate=0.05):
    """TODO: return a list of scalar losses."""
    raise NotImplementedError

_results.clear()
try:
    random.seed(1337); model=MLP(3,[4,4,1])
    losses=train(model,xs,ys)
    check("one loss per step",len(losses)==50)
    check("substantial loss reduction",losses[-1]<0.35*losses[0])
    check("losses are finite",all(math.isfinite(x) for x in losses))
except Exception as e: check("training loop",False,repr(e))
summary()


### Round C — Compare one expression with PyTorch
Recreate the calculus expression with tensors that require gradients. This is an independent oracle for your derivatives.


In [ ]:
def torch_gradient_check():
    """TODO: return [dL/da,dL/db,dL/dc] using torch autograd."""
    raise NotImplementedError

_results.clear()
try:
    tg=torch_gradient_check()
    expected=[-12.353553390593273,10.25699027111255,0.0625]
    for name,g,e in zip("abc",tg,expected):check(f"PyTorch dL/d{name}",close(g,e))
except ModuleNotFoundError: print("[SKIP] PyTorch is not installed")
except Exception as e: check("PyTorch check",False,repr(e))
summary()


### Debugging checks

1. Remove `zero_grad` and observe what happens. Explain gradient accumulation.
2. Reverse the update sign and observe the loss.
3. Try learning rates `0.001`, `0.01`, `0.05`, and `0.5`.
4. Explain why the loss is computed before `backward()`, yet parameter gradients are available afterward.
